In [2]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import statsmodels.api as sm

In [4]:
path = "/content/gdrive/MyDrive/Praktikum/Praktikum07/Data"

In [10]:
df = pd.read_csv(path + '/dataset_satelit.csv')

print("Head:")
print(df.head())
print("\nInfo:")
print(df.info())
print("\nDescribe:")
print(df.describe())

Head:
   No   Longitude  Lattitude     N     P      K    Ca    Mg      Fe      Mn  \
0   1  103.036658  -0.604417  2.64  0.15  0.415  0.51  0.31  119.96  463.23   
1   2  103.037201  -0.604689  2.75  0.17  0.568  0.76  0.58  102.63  493.81   
2   3  103.036359  -0.603012  1.77  0.12  0.339  0.49   0.6  107.37  460.93   
3   4  103.036950  -0.603219  2.30  0.15  0.460  0.74  0.67   96.02  338.17   
4   5  103.036802  -0.601969  2.05  0.14  0.308  0.64  0.72   87.01  384.33   

   ...      b1  Sigma_VV  Sigma_VH      plia       lia      iafe  gamma0_vv  \
0  ...  0.0433   0.18183   0.04461  35.74446  35.79744  35.41161    0.22331   
1  ...  0.0465   0.22079   0.04640  35.12096  35.14591  35.41510    0.27116   
2  ...  0.0417   0.18926   0.03992  35.07724  35.07730  35.41135    0.23242   
3  ...  0.0367   0.14769   0.03622  36.08078  36.08469  35.41583    0.18138   
4  ...  0.0361   0.18205   0.03797  32.68855  32.69293  35.41592    0.22359   

   gamma0_vh  beta0_vv  beta0_vh  
0    0.05

In [11]:
df.columns

Index(['No', 'Longitude', 'Lattitude', 'N', 'P', 'K', 'Ca', 'Mg', 'Fe', 'Mn',
       'Cu', 'Zn', 'B', 'b12', 'b11', 'b9', 'b8a', 'b8', 'b7', 'b6', 'b5',
       'b4', 'b3', 'b2', 'b1', 'Sigma_VV', 'Sigma_VH', 'plia', 'lia', 'iafe',
       'gamma0_vv', 'gamma0_vh', 'beta0_vv', 'beta0_vh'],
      dtype='object')

In [12]:
target = 'Fe'

In [13]:
if 'Mg' in df.columns:
    df['Mg'] = pd.to_numeric(df['Mg'], errors='coerce')

In [15]:
feature_cols = ['b2','b3','b4','b8','b11','Sigma_VV','Sigma_VH']

In [16]:
missing_features = [c for c in feature_cols if c not in df.columns]
if len(missing_features) > 0:
    raise ValueError(f"Kolom fitur tidak ditemukan: {missing_features}. Cek nama kolom di dataset.")

if target not in df.columns:
    raise ValueError(f"Kolom target '{target}' tidak ada di dataset.")

In [17]:
df_model = df[feature_cols + [target]].copy()
df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_cols + [target])

In [18]:
for c in feature_cols + [target]:
    df_model[c] = pd.to_numeric(df_model[c], errors='coerce')
df_model = df_model.dropna(subset=feature_cols + [target])

In [19]:
X = df_model[feature_cols]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nData siap. n_train={len(X_train)}, n_test={len(X_test)}")
print(f"Target yang dipakai: {target}")



Data siap. n_train=475, n_test=119
Target yang dipakai: Fe


In [20]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [22]:
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("\n=== Evaluasi (LinearRegression) ===")
print(f"R2   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")



=== Evaluasi (LinearRegression) ===
R2   : 0.0799
RMSE : 62.5034


In [23]:
coeff = pd.DataFrame({
    'Fitur': feature_cols,
    'Koefisien': model.coef_
})
print("\nKoefisien regresi:")
print(coeff)


Koefisien regresi:
      Fitur   Koefisien
0        b2  171.198417
1        b3  -68.802012
2        b4    0.913021
3        b8    2.442565
4       b11 -129.666366
5  Sigma_VV  155.643119
6  Sigma_VH -561.090404


In [24]:
X_train_sm = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_sm).fit()
print("\n=== Ringkasan OLS (statsmodels) ===")
print(ols_model.summary())


=== Ringkasan OLS (statsmodels) ===
                            OLS Regression Results                            
Dep. Variable:                     Fe   R-squared:                       0.075
Model:                            OLS   Adj. R-squared:                  0.061
Method:                 Least Squares   F-statistic:                     5.424
Date:                Sun, 09 Nov 2025   Prob (F-statistic):           5.35e-06
Time:                        13:48:22   Log-Likelihood:                -2538.2
No. Observations:                 475   AIC:                             5092.
Df Residuals:                     467   BIC:                             5126.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         7